<a href="https://colab.research.google.com/github/Amal1703/AI_ML_football/blob/main/stage_1_train_YOLO_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Turn on GPU acceleration

In [ ]:
# In Colab:
# Click Runtime
# Then click Change runtime type
# Under Hardware accelerator, select GPU T4
# Click Save


# Install dependencies

In [2]:
!uv pip install roboflow
!uv pip install ultralytics

Using Python 3.13.16 environment at: /usr
Resolved 32 packages in 382ms
Prepared 4 packages in 188ms
Uninstalled 1 package in 5ms
Installed 4 packages in 8ms
 + filetype==1.2.0
 + pillow-avif-plugin==1.6.0
 + roboflow==1.7.4
 - typer==0.27.3
 + typer==0.25.1
Using Python 3.13.16 environment at: /usr
Resolved 58 packages in 602ms
Prepared 3 packages in 1.18s
Installed 3 packages in 39ms
 + ultralytics==8.4.218
 + ultralytics-platform==0.1.121
 + ultralytics-thop==2.2.5


# Import libraries

In [16]:
from roboflow import Roboflow
from ultralytics import YOLO

from pathlib import Path
import requests
import zipfile
import os
from IPython.display import HTML
from base64 import b64encode
import subprocess
import imageio_ffmpeg
import glob

# Download dataset

In [10]:
# Download the dataset from Roboflow (yolov11)
# 1. from https://universe.roboflow.com/roboflow-jvuqo/football-players-detection-3zvbc/dataset/20
# 2. ON Popular Download Formats, cliqu on yolov1
# 3. clic on Download dataset and copy the python code (code au dessous) to dwonlod data

rf = Roboflow(api_key="waNECdEukH3W2rtXwlET")
project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
version = project.version(20)
dataset = version.download("yolov11")

loading Roboflow workspace...
loading Roboflow project...


KeyboardInterrupt: 

# Train YOLO 11 model

In [7]:
!yolo task=detect mode=train data={dataset.location}/data.yaml model=yolo11n.pt epochs=140 imgsz=640 val=True patience=15

Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/football-players-detection-20/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=140, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=None, opset=None, 

# Test the input data (video) using YOLO model

In [14]:
# Download data

repo = "Amal1703/AI_ML_football"
files = [
   "input_data.zip",
]

dl_dir = Path("colab")
dl_dir.mkdir(exist_ok=True)

def download(name):
    out = dl_dir / name
    for base in ("https://media.githubusercontent.com/media", "https://raw.githubusercontent.com"):
        url = f"{base}/{repo}/main/{name}" if "media" in base else f"{base}/{repo}/main/{name}"
        try:
            with requests.get(url, stream=True, allow_redirects=True, timeout=60) as r:
                r.raise_for_status()
                with open(out, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        f.write(chunk)
            print(f"OK   {name}  ({out.stat().st_size/1e6:.1f} MB) <- {url}")
            return out
        except requests.RequestException as e:
            print(f"Failed {url}: {e}")
    print(f"!! Could not download {name}")
    return None

downloaded = [p for p in (download(n) for n in files) if p]

# Bundle everything into colab.zip
with zipfile.ZipFile("colab.zip", "w", zipfile.ZIP_STORED) as z:  # STORED: files are already compressed
    for p in downloaded:
        z.write(p, arcname=p.name)

print("colab.zip created:", Path("colab.zip").stat().st_size / 1e6, "MB")

Failed https://media.githubusercontent.com/media/Amal1703/AI_ML_football/main/input_data.zip: 404 Client Error: Not Found for url: https://media.githubusercontent.com/media/Amal1703/AI_ML_football/main/input_data.zip
OK   input_data.zip  (20.3 MB) <- https://raw.githubusercontent.com/Amal1703/AI_ML_football/main/input_data.zip
colab.zip created: 20.34202 MB


In [15]:
with zipfile.ZipFile('/content/colab/input_data.zip', 'r') as zip_ref:
   zip_ref.extractall('/content/colab')

In [9]:
# load model
model = YOLO('/content/runs/detect/train/weights/best.pt')

In [18]:
results =  next(model.predict(
    source="/content/colab/input_data/08fd33_4.mp4",
    save=True,
    project=os.getcwd(),   # dossier racine de sortie
    name='output_data_after_yolo_prediction',      # sous-dossier
    exist_ok=True ,             # ne pas créer "output_data_after_yolo_prediction2", etc.
    stream=True  ,
    verbose=False
))

for r in results:
    pass

In [25]:
ffmpeg_exe = imageio_ffmpeg.get_ffmpeg_exe()

# Retrieve the .avi path video and convert it to mp4
avi_path = glob.glob(r.save_dir + "/*.avi")[0]
mp4_path = avi_path.replace(".avi", ".mp4")

# Conversion with ffmpeg
subprocess.run([ffmpeg_exe, "-y", "-i", avi_path, "-vcodec", "libx264", mp4_path])

# Display
mp4 = open(mp4_path, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f'<video width=500 controls><source src="{data_url}" type="video/mp4"></video>')